<a href="https://colab.research.google.com/github/mbahramii/conveyor-stone-detector/blob/main/Stone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()  # یه پنجره باز میشه برای انتخاب فایل


Saving frames.zip to frames.zip


In [ ]:
import zipfile

with zipfile.ZipFile('/content/frames.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("✅ Done!")

✅ Done!


In [ ]:
import os

files = os.listdir('/content/')
print(files)


['.config', 'frames', 'frames.zip', 'sample_data']


In [ ]:
import os

frames = os.listdir('/content/frames/')
print(f"تعداد فریم‌ها: {len(frames)}")
print("چند نمونه:", frames[:5])

تعداد فریم‌ها: 158
چند نمونه: ['frame_0100.jpg', 'frame_0141.jpg', 'frame_0005.jpg', 'frame_0053.jpg', 'frame_0107.jpg']


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# 0. تنظیمات — فقط اینجا رو عوض کن
# ═══════════════════════════════════════════════════════════════
FRAMES_DIR = "/content/frames"   # مسیر فریم‌ها
OUTPUT_DIR = "/content/output"   # مسیر خروجی
ROI_Y_RATIO = 0.55               # درصد از بالا (ناحیه غلتک)
ROI_X_RATIO = 0.48               # درصد از چپ (ناحیه غلتک)
PERCENTILE   = 97                # حساسیت تشخیص (بالاتر = سخت‌گیرتر)
CLOSE_SIZE   = 35                # اندازه morphology closing
MIN_AREA     = 400               # حداقل مساحت سنگ (px²)
MAX_AR       = 3.0               # حداکثر aspect ratio (فیلتر اشکال باریک)

# ═══════════════════════════════════════════════════════════════
# 1. لود فریم‌ها
# ═══════════════════════════════════════════════════════════════
frames_dir = Path(FRAMES_DIR)
frame_files = sorted(list(frames_dir.glob("*.jpg")) +
                     list(frames_dir.glob("*.png")))

assert len(frame_files) > 0, f"هیچ فریمی در {FRAMES_DIR} پیدا نشد!"
print(f"✅ {len(frame_files)} فریم لود شد")

frames_bgr  = [cv2.imread(str(f)) for f in frame_files]
frames_gray = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY).astype(np.float32)
               for f in frames_bgr]

h, w = frames_gray[0].shape
ROI_Y = int(h * ROI_Y_RATIO)
ROI_X = int(w * ROI_X_RATIO)
n     = len(frames_gray)

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"   رزولوشن: {w}×{h}  |  ROI از ({ROI_X},{ROI_Y}) به ({w},{h})")

# ═══════════════════════════════════════════════════════════════
# 2. تابع تشخیص
# ═══════════════════════════════════════════════════════════════
def detect_stone(idx):
    """
    Returns (rect, contour) or (None, None)
    rect = cv2.minAreaRect → ((cx,cy),(rw,rh),angle)
    """
    # heatmap: تفاوت فریم با میانگین 8 فریم اطراف
    neighbors = [frames_gray[max(0, min(n-1, idx+d))]
                 for d in [-4, -3, -2, -1, 1, 2, 3, 4]]
    ref  = np.mean(neighbors, axis=0)
    diff = cv2.GaussianBlur(
               np.abs(frames_gray[idx] - ref), (11, 11), 2)

    # فقط ROI پایین-راست
    roi = diff.copy()
    roi[:ROI_Y, :] = 0
    roi[:, :ROI_X] = 0

    vals = roi[roi > 0]
    if len(vals) == 0:
        return None, None

    # threshold روی top X% روشن‌ترین پیکسل‌ها
    thr  = np.percentile(vals, PERCENTILE)
    mask = (roi > thr).astype(np.uint8) * 255

    # morphology
    mask = cv2.morphologyEx(
               mask, cv2.MORPH_OPEN,
               cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    mask = cv2.morphologyEx(
               mask, cv2.MORPH_CLOSE,
               cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                         (CLOSE_SIZE, CLOSE_SIZE)))

    # بزرگترین contour با aspect ratio معقول
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
    cnts = sorted([c for c in cnts if cv2.contourArea(c) > MIN_AREA],
                  key=cv2.contourArea, reverse=True)
    if not cnts:
        return None, None

    for cnt in cnts:
        rect = cv2.minAreaRect(cnt)
        rw, rh = rect[1]
        if rw == 0 or rh == 0:
            continue
        if max(rw, rh) / min(rw, rh) < MAX_AR:
            return rect, cnt

    return cv2.minAreaRect(cnts[0]), cnts[0]

# ═══════════════════════════════════════════════════════════════
# 3. اجرا روی همه فریم‌ها
# ═══════════════════════════════════════════════════════════════
print("\n🔍 در حال تشخیص سنگ ...")
results = []
thumbs  = []

for idx in range(n):
    rect, cnt = detect_stone(idx)
    vis = frames_bgr[idx].copy()

    info = {"frame": str(frame_files[idx].name), "stone": None}

    if rect is not None:
        box = np.intp(cv2.boxPoints(rect))
        cx, cy = int(rect[0][0]), int(rect[0][1])
        rw, rh = int(rect[1][0]), int(rect[1][1])
        ang    = rect[2]

        info["stone"] = dict(center=[cx, cy], size=[rw, rh],
                             angle=round(ang, 1))

        # overlay
        ov = vis.copy()
        cv2.drawContours(ov, [cnt], -1, (0, 80, 255), -1)
        cv2.addWeighted(ov, 0.30, vis, 0.70, 0, vis)

        # کادر زرد
        cv2.drawContours(vis, [box], 0, (0, 220, 255), 3)
        cv2.circle(vis, (cx, cy), 6, (0, 220, 255), -1)
        cv2.putText(vis,
                    f"{frame_files[idx].stem}  {rw}x{rh}px  {ang:.0f}deg",
                    (cx-60, cy-14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 255), 2)
    else:
        cv2.putText(vis, f"{frame_files[idx].stem}  NO STONE",
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 0, 255), 2)

    results.append(info)
    thumbs.append(cv2.resize(vis, (420, 260)))

    if idx % 10 == 0:
        status = "STONE ✅" if rect else "---"
        print(f"   {frame_files[idx].name}: {status}")

# ═══════════════════════════════════════════════════════════════
# 4. موزاییک خروجی
# ═══════════════════════════════════════════════════════════════
cols = 4
while len(thumbs) % cols:
    thumbs.append(np.zeros((260, 420, 3), np.uint8))

rows   = [np.hstack(thumbs[i:i+cols]) for i in range(0, len(thumbs), cols)]
mosaic = np.vstack(rows)

mosaic_path = f"{OUTPUT_DIR}/mosaic.jpg"
cv2.imwrite(mosaic_path, mosaic)

# ═══════════════════════════════════════════════════════════════
# 5. نمایش موزاییک در Colab
# ═══════════════════════════════════════════════════════════════
from google.colab.patches import cv2_imshow
print("\n📊 موزاییک نتایج:")
cv2_imshow(mosaic)

# ═══════════════════════════════════════════════════════════════
# 6. آمار
# ═══════════════════════════════════════════════════════════════
detected = [r for r in results if r["stone"]]
print(f"\n{'─'*45}")
print(f"  کل فریم‌ها      : {len(results)}")
print(f"  سنگ تشخیص داده : {len(detected)}")
print(f"  دقت             : {len(detected)/len(results)*100:.0f}%")
if detected:
    ws   = [r["stone"]["size"][0] for r in detected]
    hs   = [r["stone"]["size"][1] for r in detected]
    angs = [r["stone"]["angle"]   for r in detected]
    print(f"  عرض سنگ  avg={np.mean(ws):.0f}  range={min(ws)}–{max(ws)} px")
    print(f"  ارتفاع   avg={np.mean(hs):.0f}  range={min(hs)}–{max(hs)} px")
    print(f"  زاویه    avg={np.mean(angs):.1f}  range={min(angs):.1f}–{max(angs):.1f} deg")
print(f"{'─'*45}")
print(f"\n💾 موزاییک ذخیره شد: {mosaic_path}")